In [1]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import sklearn as skl
import gsw
import cartopy.crs as ccrs
from scipy.interpolate import interp1d
import os
import warnings
import joblib
warnings.filterwarnings('ignore')

In [2]:
os.system('echo $USER > userid')
usrid=np.genfromtxt('userid',dtype='<U16')
os.system(f'mkdir -p /glade/derecho/scratch/{usrid}/ML4O2_temp/')
os.system(f'mkdir -p /glade/derecho/scratch/{usrid}/ML4O2_results/')
os.system(f'mkdir -p /glade/derecho/scratch/{usrid}/WOD18_OSDCTD/')
#
# version information
# ver = np.genfromtxt('data_XXX.txt',dtype='U11').tolist()
ver = '1.2.1.6.2B.4'
date1='07082025' # Set this for saving today's date. Usually date1=today's date
date2='07082025' # Set alternative date for re-running previous results
rerun = False    # indicate again whether you are re-running previous results
#
dirout=f'/glade/derecho/scratch/{usrid}/ML4O2_temp/'
#
print(f'-----------------------------------------------')
print(f' Machine Learning For Dissolved Oxygen (ML4O2) ')
print(f' version{ver} date:{date1}')
print(f'-----------------------------------------------')

-----------------------------------------------
 Machine Learning For Dissolved Oxygen (ML4O2) 
 version1.2.1.6.2B.4 date:07082025
-----------------------------------------------


In [3]:
selection = ver.split('.')
basin = ['Atlantic','Pacific','Indian','Southern','Arctic','Mediterranean']
#
if selection[0] == '1':
    print('Random Forst algorithm will be used.')
    alg = 'RF'
elif selection[0] == '20':
    print('Neural Network algorithm will be used.')
    alg = 'NN'
elif selection[0] == '31':
    print('Gradient Boosting algorithm will be used.')
    alg = 'GB'
else:
    print('error - incorrect algorithm type')
#
if selection[1] == '1':
    print('Ship-based O2 data will be used. ')
elif selection[1] == '2':
    print('Ship-based and Argo-O2 data will be used. ')
else:
    print('error - incorrect input data type')
#
if selection[2] == '1':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
elif selection[2] == '2':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
elif selection[2] == '3':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
elif selection[2] == '4':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
elif selection[2] == '5':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
elif selection[2] == '6':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
else:
    print('error - incorrect O2 data type')
#
if selection[3] == '1':
    print('EN4 dataset will be used for T/S input. ')
    endyear=2021
    if selection[1]=='1':
        dirinput = f'/glade/derecho/scratch/qzhang459/ML4O2/input_20250708/'
elif selection[3] == '2':
    print('ORAS4 dataset will be used for T/S input. ')
    endyear=2018
elif selection[3] == '3':
    print('SODA3.4.2 dataset will be used for T/S input. ')
elif selection[3] == '4':
    print('EN4 (C14) dataset will be used for T/S input. ')
    endyear=2023
    if selection[1]=='1':
        #dirinput = f'/glade/work/ito/ML4O2/apr2025/input_202504_ship/'
        dirinput = f'/glade/derecho/scratch/qzhang459/ML4O2/input_20250708/'
    elif selection[1]=='2':
        #dirinput = f'/glade/work/ito/ML4O2/apr2025/input_202504_noadj/'
        dirinput = f'/glade/derecho/scratch/qzhang459/ML4O2/input_20250708/'
    elif selection[1]=='3':
        dirinput = f'/glade/derecho/scratch/qzhang459/ML4O2/input_20250708/'
    Nlev=63
elif selection[3] == '6':
    print('IPSL dataset will be used for T/S input. ')
    dirinput = f'/glade/derecho/scratch/qzhang459/EMU2/CMIP6/IPSL-CM6A-LR/input'
    endyear=2023
else:
    print('error - incorrect T/S data type')
#
if selection[4] == '1':
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, month')
elif selection[4] == '2':
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month)')
elif selection[4] == '2B':
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month) with O2bias')
elif selection[4] == '3':
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month), sigma')
elif selection[4] == '4':
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month), sigma, N2')
#NEW
elif selection[4] == '5AN':
    print('Predictor variables include T, S, lon, lat, depth (pressure), cos(month), sin(month) with O2ano')
elif selection[4] == '5B':
    print('Predictor variables include T, S, lon, lat, depth (pressure), cos(month), sin(month) with O2bias')

elif selection[4] == '6':
    print('Predictor variables include T, S, lon, lat, depth (pressure), GMT, cos(month), sin(month)')
elif selection[4] == '6A':
    print('Predictor variables include T, S, lon, lat, depth (pressure), GMT, cos(month), sin(month) with AOU')
elif selection[4] == '6B':
    print('Predictor variables include T, S, lon, lat, depth (pressure), GMT, cos(month), sin(month) with O2bias')
elif selection[4] == '60':
    print('Predictor variables include T, S, lon, lat, depth (pressure), cos(month), sin(month) with AOU')
elif selection[4] == '8':
    print('Predictor variables include T, S, lon, lat, depth (pressure), GMT, O2sol, cos(month), sin(month)')
#NEW
else:
    print('error - incorrect predictor variable type')
#
if selection[5] == '1':
    print('Hyperparameter set is optimized via K-fold CV')
elif selection[5] == '2':
    print('A pre-set hyperparameter set is used')
elif selection[5] == '4':
    print('New K-fold cross validation')
else:
    print('error - incorrect hyperparameter type')

Random Forst algorithm will be used.
Ship-based and Argo-O2 data will be used. 
Atlantic Ocean will be mapped
IPSL dataset will be used for T/S input. 
Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month) with O2bias
New K-fold cross validation


In [4]:
diro = f'/glade/derecho/scratch/{usrid}/WOD18_OSDCTD/'
if selection[3] == 1:
    dirf = '/glade/campaign/univ/ugit0034/EN4/L09_20x180x360/'
elif selection[3] == 2:
    dirf = '/glade/campaign/univ/ugit0034/ORAS4/TSN2/'
#
dirin = '/glade/campaign/univ/ugit0034/WOD18_OSDCTD/'
fargo = '/glade/campaign/univ/ugit0034/bgcargo/o2_Global_ARGO_Type12_47lev.nc'
fosd='_1x1bin_osd_'
fctd='_1x1bin_ctd_'
fmer='_1x1bin_osdctd_'
var=['o2','TSN2']
os.system('mkdir -p '+diro)
os.system('mkdir -p '+diro+'temp')

0

In [5]:
ds=xr.open_dataset(dirin+var[0]+fmer+str(1965)+'.nc')
Z=ds.depth.to_numpy()
Nz=np.size(Z)


# In[6]:


# select analysis period
# do not change the start year from 1965 (this is when Carpenter 1965 established modern Winkler method)
yrs=np.arange(1965,endyear,1)

In [6]:
dsm=xr.open_dataset('/glade/campaign/univ/ugit0034/wod18/basin_mask_01.nc')
#
if selection[2] == '1':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='atlantic'
elif selection[2] == '2':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='pacific'
elif selection[2] == '3':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='indian'
elif selection[2] == '4':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='southern'
elif selection[2] == '5':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='arctic'
elif selection[2] == '6':
    print(basin[int(selection[2])-1]+' Ocean will be mapped')
    bname0='mediterranean'
else:
    print('error - incorrect O2 data type')
#
if selection[3]=='1':
    if selection[1]=='2':
        doa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/o20_{bname0}_1x1_47lev.npy')
        dta1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/t0_{bname0}_1x1_47lev.npy')
        dsa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/s0_{bname0}_1x1_47lev.npy')
        xx1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/lon0_{bname0}_1x1_47lev.npy')
        yy1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/lat0_{bname0}_1x1_47lev.npy')
        zz1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/depth0_{bname0}_1x1_47lev.npy')
        tt1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/time0_{bname0}_1x1_47lev.npy')
        tc1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/month0_{bname0}_1x1_47lev.npy')
        dsga1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/sigma0_{bname0}_1x1_47lev.npy')
        dn2a1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4_offsetWOA/N20_{bname0}_1x1_47lev.npy')
    elif selection[1]=='1':
        doa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/o20_{bname0}_1x1_47lev_ship.npy')
        dta1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/t0_{bname0}_1x1_47lev_ship.npy')
        dsa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/s0_{bname0}_1x1_47lev_ship.npy')
        xx1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/lon0_{bname0}_1x1_47lev_ship.npy')
        yy1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/lat0_{bname0}_1x1_47lev_ship.npy')
        zz1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/depth0_{bname0}_1x1_47lev_ship.npy')
        tt1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/time0_{bname0}_1x1_47lev_ship.npy')
        tc1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/month0_{bname0}_1x1_47lev_ship.npy')
        dsga1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/sigma0_{bname0}_1x1_47lev_ship.npy')
        dn2a1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/EN4/N20_{bname0}_1x1_47lev_ship.npy')
    if (selection[4]=='6')&(selection[1]=='1'):
        doa1 = np.load(f'{dirinput}/o20_{bname0}_1x1_47lev.npy')
        dta1 = np.load(f'{dirinput}/t0_{bname0}_1x1_47lev.npy')
        dsa1 = np.load(f'{dirinput}/s0_{bname0}_1x1_47lev.npy')
        xx1 = np.load(f'{dirinput}/lon0_{bname0}_1x1_47lev.npy')
        yy1 = np.load(f'{dirinput}/lat0_{bname0}_1x1_47lev.npy')
        zz1 = np.load(f'{dirinput}/depth0_{bname0}_1x1_47lev.npy')
        tt1 = np.load(f'{dirinput}/time0_{bname0}_1x1_47lev.npy')
        tc1 = np.load(f'{dirinput}/month0_{bname0}_1x1_47lev.npy') 
        GMT1 = np.load(f'{dirinput}/GMT0_{bname0}_1x1_47lev.npy') 
#
elif selection[3]=='2':
    if selection[1]=='2':
        doa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/o20_{bname0}_1x1_47lev.npy')
        dta1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/t0_{bname0}_1x1_47lev.npy')
        dsa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/s0_{bname0}_1x1_47lev.npy')
        xx1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/lon0_{bname0}_1x1_47lev.npy')
        yy1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/lat0_{bname0}_1x1_47lev.npy')
        zz1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/depth0_{bname0}_1x1_47lev.npy')
        tt1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/time0_{bname0}_1x1_47lev.npy')
        tc1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/month0_{bname0}_1x1_47lev.npy')
        dsga1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/sigma0_{bname0}_1x1_47lev.npy')
        dn2a1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/N20_{bname0}_1x1_47lev.npy')
    elif selection[1]=='1':
        doa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/o20_{bname0}_1x1_47lev_ship.npy')
        dta1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/t0_{bname0}_1x1_47lev_ship.npy')
        dsa1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/s0_{bname0}_1x1_47lev_ship.npy')
        xx1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/lon0_{bname0}_1x1_47lev_ship.npy')
        yy1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/lat0_{bname0}_1x1_47lev_ship.npy')
        zz1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/depth0_{bname0}_1x1_47lev_ship.npy')
        tt1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/time0_{bname0}_1x1_47lev_ship.npy')
        tc1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/month0_{bname0}_1x1_47lev_ship.npy')
        dsga1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/sigma0_{bname0}_1x1_47lev_ship.npy')
        dn2a1 = np.load(f'/glade/campaign/univ/ugit0034/ML4O2/input_202404/ORAS4/N20_{bname0}_1x1_47lev_ship.npy')
elif selection[3]=='6':
    doa1 = np.load(f'{dirinput}/o2b0_{bname0}_1x1_63lev.npy') # O2bias!!
    # doa1 = np.load(f'{dirinput}/o20_{bname0}_1x1_63lev.npy')
    # aou1 = np.load(f'{dirinput}/aou0_{bname0}_1x1_63lev.npy')
    dta1 = np.load(f'{dirinput}/t0_{bname0}_1x1_63lev.npy')
    dsa1 = np.load(f'{dirinput}/s0_{bname0}_1x1_63lev.npy')
    xx1 = np.load(f'{dirinput}/lon0_{bname0}_1x1_63lev.npy')
    yy1 = np.load(f'{dirinput}/lat0_{bname0}_1x1_63lev.npy')
    zz1 = np.load(f'{dirinput}/depth0_{bname0}_1x1_63lev.npy')
    tt1 = np.load(f'{dirinput}/time0_{bname0}_1x1_63lev.npy')
    tc1 = np.load(f'{dirinput}/month0_{bname0}_1x1_63lev.npy')
    GMT1 = np.load(f'{dirinput}/GMT0_{bname0}_1x1_63lev.npy')
    # o2sol1 = np.load(f'{dirinput}/o2sol0_{bname0}_1x1_63lev.npy')

# In[8]:

Nsample = np.size(doa1)
print(Nsample)

Atlantic Ocean will be mapped
3329584


In [7]:
if selection[4] == '1':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, tt1, tc1])
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, month')
elif selection[4] == '2' or '2B':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, tt1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12)])
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month)')
elif selection[4] == '3':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, tt1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12), dsga1])
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month), sigma')
elif selection[4] == '4':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, tt1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12), dsga1, dn2a1])
    print('Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month), sigma, N2')
elif selection[4] == '5':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12)])
    print('Predictor variables include T, S, lon, lat, depth (pressure), cos(month), sin(month)')
elif selection[4] == '5B':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12)])
    print('Predictor variables include T, S, lon, lat, depth (pressure), cos(month), sin(month) with O2bias')
elif selection[4] == '6':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, GMT1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12)])
    print('Predictor variables include T, S, lon, lat, depth (pressure), GMT, cos(month), sin(month)')
elif selection[4] == '6B':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, GMT1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12)])
    print('Predictor variables include T, S, lon, lat, depth (pressure), GMT, cos(month), sin(month) with O2bias')
elif selection[4] == '6A':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, GMT1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12)])
    print('Predictor variables include T, S, lon, lat, depth (pressure), GMT, cos(month), sin(month) with AOU')
elif selection[4] == '60':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12)])
    print('Predictor variables include T, S, lon, lat, depth (pressure), cos(month), sin(month) with AOU')
elif selection[4] == '8':
    X = np.array([dsa1, dta1, xx1, yy1, zz1, GMT1, o2sol1, np.cos(2*np.pi*tc1/12), np.sin(2*np.pi*tc1/12)])
    print('Predictor variables include T, S, lon, lat, depth (pressure), GMT, o2sol, cos(month), sin(month)')
else:
    print('error - incorrect predictor variable type')

Predictor variables include T, S, lon, lat, depth (pressure), year, cos(month), sin(month)


In [10]:
y = doa1
#
Xm = np.mean(X,axis=1)
Xstd = np.std(X,axis=1)
#
N=np.size(y)
# normalize x and y
Xa = (X.T - Xm)/Xstd
ym = np.mean(y)
ystd = np.std(y)
ya = (y-ym)/ystd
#
np.savez(dirout+f'ML_params_v{ver}.npz',Xm=Xm,Xstd=Xstd,ym=ym,ystd=ystd)

In [11]:
if rerun==False:
    yr_drop = np.random.choice(yrs,11,replace=False)
    print(yr_drop)

# group these years together into a single array
if rerun==False:
    yr1=np.round(tt1/12+1965)
    ind=(yr1==int(yr_drop[0]))
    N=np.sum(ind)
    for n in np.arange(1,11,1):
        tmp=(yr1==yr_drop[n])
        ind=(ind==True)|(tmp==True)
        N=N+np.sum(tmp)
    #print(N,np.sum(ind))
    print(f'the count of data point (bins) = {yr1.size}')
    print(f'the count of test data (bins) = {N}, which is {N/yr1.size*100}%')

# Assemble into input data (train/test) and save it for record
if rerun==False:
    ind1 = (ind==False)
    X_train = Xa[ind1,:]
    X_test = Xa[ind,:]
    y_train = ya[ind1]
    y_test = ya[ind]
    print(f'the count of train data point (bins) = {y_train.size}')
    np.savez(dirout+f'train_test_v{ver}_{date1}.npz',X_train=X_train,X_test=X_test,
             y_train=y_train,y_test=y_test,yr_drop=yr_drop)

[2017 2012 1985 2001 1993 1977 2019 1968 1996 1987 2021]
the count of data point (bins) = 3329584
the count of test data (bins) = 661983, which is 19.8818531083763%
the count of train data point (bins) = 2667601


In [7]:
tmp=np.load(dirout+f'train_test_v{ver}_{date2}.npz')
X_train = tmp['X_train']
X_test  = tmp['X_test']
y_train = tmp['y_train']
y_test  = tmp['y_test']
#
tmp = np.load(dirout+f'ML_params_v{ver}.npz')
Xstd = tmp['Xstd']
Xm   = tmp['Xm']
#
#
if (selection[4] == '5B')|(selection[4] == '6')|(selection[4] == '6B')|(selection[4] == '5'):
    ttmp0 = tt1[ind1]
else:
    ttmp0 = X_train[:,5]*Xstd[5]+Xm[5]
#
yr1 = ttmp0/12+1965

# Calculate the Decadal Group K-fold
tbnds=[1965,1977,1989,2000,2012,2022]
Kval = 5
#yr1=X[5,ind1]/12+1965
print(f'The total count of data points = {yr1.size}')

The total count of data points = 2667601


In [8]:
for n in range(5):
    K_test=(yr1>=tbnds[n])&(yr1<tbnds[n+1])
    K_train=(K_test==False)
    X_trainK = X_train[K_train,:]
    X_testK = X_train[K_test,:]
    y_trainK = y_train[K_train]
    y_testK = y_train[K_test]
    # check
    print(f'N,train = {y_train.size}, Group {n} train size = {y_trainK.size}, Group {n} test size = {y_testK.size}, {y_testK.size/y_train.size*100}%')

N,train = 2667601, Group 0 train size = 2164214, Group 0 test size = 503387, 18.870400783325543%
N,train = 2667601, Group 1 train size = 1999050, Group 1 test size = 668551, 25.06188144328931%
N,train = 2667601, Group 2 train size = 2340435, Group 2 test size = 327166, 12.264427851091673%
N,train = 2667601, Group 3 train size = 2307089, Group 3 test size = 360512, 13.514464869371395%
N,train = 2667601, Group 4 train size = 2033646, Group 4 test size = 633955, 23.764985843085228%


In [9]:
# RF_parameters = {'min_samples_split': [8,16,32,64,128],'min_samples_leaf': [10, 20, 30]}
# RF_parameters = {'min_samples_split':[128,256,512,1024,2048,4096],'max_features':[2,3,4]}
RF_parameters = {'min_samples_split':[2,4,8,16,32,64],'max_features':[2,3,5]}
NN_parameters = {'hidden_layer_sizes':[[5,5],[10,10],[20,20],[40,40],
                                       [20,10],[20,15,10,5],[20,15,10,5,5,5]],'alpha':[.001, .01, .1]}

In [10]:
def train_K(k):
    if alg =='RF':
        from sklearn.ensemble import RandomForestRegressor
        mxf=RF_parameters['max_features'][parm2]
        msp=RF_parameters['min_samples_split'][parm1]
        msl=5
        nest=500
        regr=RandomForestRegressor(n_jobs=-1,n_estimators=nest,min_samples_split=msp,
                                   min_samples_leaf=msl,max_features=mxf)
        K_test=(yr1>=tbnds[k])&(yr1<tbnds[k+1])
        K_train=(K_test==False)
        X_trainK = X_train[K_train,:]
        X_testK = X_train[K_test,:]
        y_trainK = y_train[K_train]
        y_testK = y_train[K_test]
        regr.fit(X_trainK, y_trainK)
        y_est = regr.predict(X_testK)
        np.savez(dirout+f'RFtest_pred_v{ver}_cv{k}_{parm1}_{parm2}.npz',Xtest=X_testK,test=y_testK,est=y_est)
        filename = dirout+f'algorithm_v{ver}_cv{k}_{parm1}_{parm2}.sav'
        joblib.dump(regr, filename)
    elif alg == 'NN':
        from sklearn.neural_network import MLPRegressor
        hls=NN_parameters['hidden_layer_sizes'][parm1]
        alp=NN_parameters['alpha'][parm2]
        regr=MLPRegressor(max_iter=1000,hidden_layer_sizes=hls,alpha=alp)
        K_test=(yr1>=tbnds[k])&(yr1<tbnds[k+1])
        K_train=(K_test==False)
        X_trainK = X_train[K_train,:]
        X_testK = X_train[K_test,:]
        y_trainK = y_train[K_train]
        y_testK = y_train[K_test]
        regr.fit(X_trainK, y_trainK)
        y_est = regr.predict(X_testK)
        np.savez(dirout+f'NNtest_pred_v{ver}_cv{k}_{parm1}_{parm2}.npz',Xtest=X_testK,test=y_testK,est=y_est)
        filename = dirout+f'algorithm_v{ver}_cv{k}_{parm1}_{parm2}.sav'
        joblib.dump(regr, filename)
    r=np.corrcoef(y_est,y_testK)
    return np.round(r[0,1]**2,4)

In [ ]:
for parm1 in range(6):
#for parm1 in [5]:
    for parm2 in range(3):
    #for parm2 in [1,2]:
        if alg =='NN':
            from multiprocessing import Pool
            if __name__ == '__main__':
                with Pool(5) as p:
                    print(p.map(train_K, [0, 1, 2, 3, 4]))
        elif alg=='RF':
            for n in range(5):
                r2=train_K(n)
                print(n,parm1,parm2,r2)
        elif alg=='GB':
            for n in range(5):
                r2=train_K(n)
                print(n,parm1,parm2,r2)

0 0 0 0.6177
1 0 0 0.6904
2 0 0 0.6322
3 0 0 0.5787
4 0 0 0.7055
0 0 1 0.6267
1 0 1 0.6965
2 0 1 0.6391
3 0 1 0.5812
4 0 1 0.7114
0 0 2 0.6238
1 0 2 0.6932
2 0 2 0.6335
4 0 2 0.7086
0 1 0 0.6194
1 1 0 0.6902
2 1 0 0.6325
3 1 0 0.5791
4 1 0 0.7057
1 1 1 0.6965
2 1 1 0.6389
3 1 1 0.5808
4 1 1 0.712
0 1 2 0.6239
1 1 2 0.6935
2 1 2 0.6347
4 1 2 0.7085
0 2 0 0.6181
1 2 0 0.6906
2 2 0 0.6323
3 2 0 0.5788
4 2 0 0.7049
0 2 1 0.6274
1 2 1 0.6964
3 2 1 0.5809
4 2 1 0.7115
0 2 2 0.624
1 2 2 0.6932
2 2 2 0.6348
3 2 2 0.5739
4 2 2 0.7087
0 3 0 0.6174
1 3 0 0.6908
2 3 0 0.6326
3 3 0 0.5785
4 3 0 0.7042


2 2 0 0.9017


3 2 0 0.9043


4 2 0 0.956


0 2 1 0.9358


...

0 5 2 0.937


1 5 2 0.9295


2 5 2 0.9025


3 5 2 0.9044


4 5 2 0.9562


In [12]:
parm1 = 4
for parm2 in range(2,3):
    #for parm2 in [1,2]:
    if alg =='NN':
        from multiprocessing import Pool
        if __name__ == '__main__':
            with Pool(5) as p:
                print(p.map(train_K, [0, 1, 2, 3, 4]))
    elif alg=='RF':
        for n in range(5):
                r2=train_K(n)
                print(n,parm1,parm2,r2)
    elif alg=='GB':
        for n in range(5):
            r2=train_K(n)
            print(n,parm1,parm2,r2)

0 4 2 0.6267
1 4 2 0.6965
2 4 2 0.6365
4 4 2 0.7102


In [13]:
parm1 = 5
for parm2 in range(3):
    #for parm2 in [1,2]:
    if alg =='NN':
        from multiprocessing import Pool
        if __name__ == '__main__':
            with Pool(5) as p:
                print(p.map(train_K, [0, 1, 2, 3, 4]))
    elif alg=='RF':
        for n in range(5):
                r2=train_K(n)
                print(n,parm1,parm2,r2)
    elif alg=='GB':
        for n in range(5):
            r2=train_K(n)
            print(n,parm1,parm2,r2)

0 5 0 0.6139
1 5 0 0.687
2 5 0 0.6259
3 5 0 0.5759
4 5 0 0.7018
0 5 1 0.6289
1 5 1 0.698
2 5 1 0.6367
3 5 1 0.582
4 5 1 0.7107
0 5 2 0.6286
1 5 2 0.6979
2 5 2 0.6375
3 5 2 0.5764
4 5 2 0.7098
